In [15]:
import os
from google.colab import drive

# 1. 일단 드라이브부터 확실하게 다시 잡습니다 (Force Remount)
drive.mount('/content/drive', force_remount=True)

# 2. 님이 말한 그 폴더 경로
folder_path = '/content/drive/MyDrive/AI_Team_Project'

print("-" * 50)
print(f"🔍 경로 확인 중: {folder_path}")

# 3. 폴더가 있는지부터 검사
if not os.path.exists(folder_path):
    print("❌ [충격] 파이썬이 'AI_Team_Project' 폴더 자체를 못 찾고 있습니다!")
    print("   혹시 폴더명에 띄어쓰기가 있나요? (예: 'AI Team Project')")
    print("   👇 내 드라이브 최상위에 있는 폴더들 중 비슷한 게 있는지 찾아보세요:")

    # 최상위 목록 출력해서 비슷한 이름 찾기
    root_files = os.listdir('/content/drive/MyDrive')
    for f in root_files:
        if "AI" in f or "Project" in f:
            print(f"   - {f}")

else:
    print("✅ 폴더는 찾았습니다! 이제 파일 목록을 뽑아봅니다.")
    print("-" * 50)

    # 4. 폴더 안의 파일 다 긁어오기
    files = os.listdir(folder_path)
    found = False
    for f in files:
        # zip 파일만 골라서 보여줌
        if f.endswith('.zip'):
            print(f"📦 [발견] {f}")
            found = True

            # 7787 들어간 거면 강조!
            if "7787" in f:
                print(f"   👉 (이거 복사해서 쓰세요!) {os.path.join(folder_path, f)}")

    if not found:
        print("❌ 폴더는 있는데 .zip 파일이 하나도 안 보입니다.")

Mounted at /content/drive
--------------------------------------------------
🔍 경로 확인 중: /content/drive/MyDrive/AI_Team_Project
✅ 폴더는 찾았습니다! 이제 파일 목록을 뽑아봅니다.
--------------------------------------------------
📦 [발견] final_dataset_preprocessed_NEW_ONLY1.2.zip
📦 [발견] final_dataset_preprocessed_old_ONLY1.2.zip
📦 [발견] data.zip
📦 [발견] yolo_dataset_new_upload.zip
📦 [발견] yolo_dataset_old_upload.zip
📦 [발견] test_images.zip
📦 [발견] YOLO_Dataset_New_Smart_Strict_7787(2).zip
   👉 (이거 복사해서 쓰세요!) /content/drive/MyDrive/AI_Team_Project/YOLO_Dataset_New_Smart_Strict_7787(2).zip


In [9]:
import zipfile
import os
from google.colab import drive
# =========================================================
# [여기만 수정하세요] 님이 가지고 있는 데이터셋 압축파일 경로
# 예: '/content/drive/MyDrive/AI_Team_Project/dataset.zip'
# =========================================================
zip_path = '/content/drive/MyDrive/AI_Team_Project/YOLO_Dataset_New_Smart_Strict_7787(2).zip'
# ▲ 만약 파일명이 다르면 위 경로를 님 파일에 맞게 고치세요!

# 압축 풀 위치 (YOLO가 읽을 폴더 이름)
extract_path = '/content/yolo_pill_ds'

# 압축 풀기 실행
if not os.path.exists(zip_path):
    print(f"❌ 파일을 못 찾겠습니다: {zip_path}")
    print("👉 구글 드라이브 경로와 파일명을 다시 확인해주세요.")
else:
    print(f"📂 압축을 풉니다... ({zip_path} -> {extract_path})")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ 압축 해제 완료! 이제 데이터가 생겼습니다.")

📂 압축을 풉니다... (/content/drive/MyDrive/AI_Team_Project/YOLO_Dataset_New_Smart_Strict_7787(2).zip -> /content/yolo_pill_ds)
✅ 압축 해제 완료! 이제 데이터가 생겼습니다.


In [23]:
import os
import glob
import yaml
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

# =========================================================
# [설정] 0.985 모델 경로
# =========================================================
model_path = '/content/drive/MyDrive/AI_Team_Project/runs/phase2_final/weights/best.pt'
root_dir = '/content/yolo_pill_ds' # 압축 푼 최상위 폴더
# =========================================================

print(f"🚀 [스마트 채점기] 파일 위치를 수색합니다...")

# 1. dataset.yaml 파일 찾기 (재귀 탐색)
found_yamls = glob.glob(os.path.join(root_dir, "**", "dataset.yaml"), recursive=True) + \
              glob.glob(os.path.join(root_dir, "**", "data.yaml"), recursive=True)

if not found_yamls:
    print("❌ 'dataset.yaml' 파일을 도저히 못 찾겠습니다.")
    print("👉 폴더 안에 yaml 파일이 아예 없는 것 같습니다.")
    exit()
else:
    dataset_yaml_path = found_yamls[0]
    print(f"✅ 설정 파일 발견: {dataset_yaml_path}")

# 2. 이미지 폴더 찾기 (재귀 탐색)
found_imgs = glob.glob(os.path.join(root_dir, "**", "images", "val", "*.jpg"), recursive=True) + \
             glob.glob(os.path.join(root_dir, "**", "images", "val", "*.png"), recursive=True)

if not found_imgs:
    print("❌ 이미지 폴더(val)를 못 찾겠습니다.")
    exit()

# 경로 확정
first_img_path = found_imgs[0]
val_img_dir = os.path.dirname(first_img_path)
val_label_dir = val_img_dir.replace('images', 'labels') # images -> labels 치환
print(f"✅ 데이터 경로 확정: {val_img_dir}")


# 3. 모델과 데이터셋의 '이름표' 로드
print("-" * 50)
model = YOLO(model_path)
model_names = model.names  # 모델이 아는 이름들 {0: 'K-123', ...}

with open(dataset_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
    dataset_names = data_config['names'] # 데이터셋의 이름들 {0: 'K-999', ...}

print(f"📌 모델 클래스 개수: {len(model_names)}")
print(f"📌 데이터셋 클래스 개수: {len(dataset_names)}")

# 4. 채점 시작
print("-" * 50)
print(f"📊 총 {len(found_imgs)}장 채점 시작 (번호 무시, 이름 비교)...")

error_data = []
correct_count = 0

for img_path in tqdm(found_imgs):
    file_name = os.path.basename(img_path)
    label_file = os.path.join(val_label_dir, os.path.splitext(file_name)[0] + ".txt")

    if not os.path.exists(label_file): continue

    # (1) 정답지 읽어서 -> 이름으로 변환
    with open(label_file, 'r') as f:
        lines = f.readlines()
        if not lines: continue
        gt_id = int(lines[0].strip().split()[0])

        # 데이터셋 기준 이름 (예: K-000123)
        # 혹시 ID가 범위를 벗어나면 에러 방지
        if gt_id in dataset_names:
            gt_name = dataset_names[gt_id]
        else:
            continue # 이상한 라벨은 패스

    # (2) 모델 예측 -> 이름으로 변환
    results = model.predict(img_path, verbose=False, conf=0.001)

    if results[0].boxes:
        top1 = results[0].boxes[0]
        pred_id = int(top1.cls.item())
        conf = top1.conf.item()

        # 모델 기준 이름 (예: K-000123)
        pred_name = model_names[pred_id]

        # (3) [핵심] 번호가 아니라 '이름'이 다르면 오답!
        if gt_name != pred_name:
            error_data.append({
                "파일명": file_name,
                "실제정답(이름)": gt_name,
                "모델예측(이름)": pred_name,
                "확신도(%)": round(conf * 100, 2),
                "실제ID": gt_id,
                "예측ID": pred_id
            })
        else:
            correct_count += 1

# 5. 결과 저장
output_path = "real_error_note.csv"
if error_data:
    df = pd.DataFrame(error_data)
    df = df.sort_values(by="확신도(%)", ascending=False)
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("\n" + "="*50)
    print(f"🔥 [최종 결과]")
    print(f" - 전체: {len(found_imgs)}장")
    print(f" - 정답: {correct_count}장")
    print(f" - 진짜 오답: {len(error_data)}장")
    print(f"📂 '{output_path}' 파일 다운로드하세요.")
    print("="*50)
else:
    print("\n🎉 와... 이름으로 비교하니까 틀린 게 하나도 없습니다! (100% 정답)")

🚀 [스마트 채점기] 파일 위치를 수색합니다...
✅ 설정 파일 발견: /content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/data.yaml
✅ 데이터 경로 확정: /content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/images/val
--------------------------------------------------
📌 모델 클래스 개수: 56
📌 데이터셋 클래스 개수: 57
--------------------------------------------------
📊 총 1575장 채점 시작 (번호 무시, 이름 비교)...


100%|██████████| 1575/1575 [00:00<00:00, 32815.53it/s]


🎉 와... 이름으로 비교하니까 틀린 게 하나도 없습니다! (100% 정답)


In [17]:
import yaml
from ultralytics import YOLO

# =========================================================
# 설정 (아까랑 동일)
# =========================================================
model_path = '/content/drive/MyDrive/AI_Team_Project/best(0.985).pt'
yaml_path = '/content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/data.yaml'
# =========================================================

print(f"🔥 [이름표 비교] 모델(57개) vs 데이터셋(56개)")

# 1. 모델 이름 로드
model = YOLO(model_path)
model_names = model.names

# 2. 데이터셋 이름 로드
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
    dataset_names = data_config['names']

# 3. 앞부분 10개만 비교 출력
print("\n{:<5} | {:<30} | {:<30}".format("ID", "모델이 아는 이름(Team)", "내 데이터셋 이름(Me)"))
print("-" * 70)

# 더 적은 개수 기준으로 비교
limit = min(len(model_names), len(dataset_names))

for i in range(limit):
    m_name = model_names[i]
    d_name = dataset_names[i]

    # 이름이 다르면 'X' 표시, 같으면 'O'
    status = "✅" if m_name == d_name else "❌"

    print("{:<5} | {:<30} | {:<30} {}".format(i, m_name, d_name, status))

    if i >= 15: break # 15개까지만 보여줌

print("-" * 70)
print(f"👉 모델에만 있는 마지막(57번째) 클래스: {model_names.get(56, '없음')}")

🔥 [이름표 비교] 모델(57개) vs 데이터셋(56개)

ID    | 모델이 아는 이름(Team)                | 내 데이터셋 이름(Me)                 
----------------------------------------------------------------------
0     | 보령부스파정 5mg                     | 보령부스파정 5mg                     ✅
1     | 뮤테란캡슐 100mg                    | 뮤테란캡슐 100mg                    ✅
2     | 일양하이트린정 2mg                    | 일양하이트린정 2mg                    ✅
3     | 기넥신에프정(은행엽엑스)(수출용)             | 기넥신에프정(은행엽엑스)(수출용)             ✅
4     | 무코스타정(레바미피드)(비매품)              | 무코스타정(레바미피드)(비매품)              ✅
5     | 알드린정                           | 알드린정                           ✅
6     | 뉴로메드정(옥시라세탐)                   | 뉴로메드정(옥시라세탐)                   ✅
7     | 에어탈정(아세클로페낙)                   | 에어탈정(아세클로페낙)                   ✅
8     | 리렉스펜정 300mg/PTP                | 리렉스펜정 300mg/PTP                ✅
9     | 아빌리파이정 10mg                    | 아빌리파이정 10mg                    ✅
10    | 다보타민큐정 10mg/병                  | 다보타민큐정 10mg/병                  ✅
11    | 써스

In [20]:
import yaml
from ultralytics import YOLO

# =========================================================
# 설정
# =========================================================
model_path = '/content/drive/MyDrive/AI_Team_Project/best(0.985).pt'
# 님 데이터셋의 yaml 파일 경로 (정확히 확인해주세요)
yaml_path = '/content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/data.yaml'
# =========================================================

print(f"🕵️ 범인 색출 시작 (어디서부터 밀렸나?)...")

# 1. 로드
model = YOLO(model_path)
model_names = model.names # 팀장님 순서 (정답지)

with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
    dataset_names = data_config['names'] # 님 순서 (오답지)

# 2. 비교
mismatch_found = False
first_mismatch_idx = -1

limit = min(len(model_names), len(dataset_names))

print("\n" + "="*60)
print("{:<5} | {:<25} | {:<25} | 상태".format("ID", "팀장 모델(Model)", "내 데이터(YAML)"))
print("="*60)

for i in range(limit):
    m_name = model_names[i]
    d_name = dataset_names[i]

    if m_name != d_name:
        print("{:<5} | {:<25} | {:<25} | ❌ <--- 여기서부터 밀림!".format(i, m_name, d_name))
        mismatch_found = True
        first_mismatch_idx = i
        break # 범인 찾았으니 중단
    else:
        # 15번 이후로도 잘 맞는지 확인용 (로그 너무 길까봐 생략하려다 5개 단위로만 출력)
        if i > 15 and i % 5 == 0:
             print("{:<5} | {:<25} | {:<25} | ✅".format(i, m_name, d_name))

if not mismatch_found:
    print("-" * 60)
    print("어라? 이름은 끝까지 똑같은데요? (그럼 마지막 57번째만 없는 겁니다)")
else:
    print("-" * 60)
    print(f"🚨 찾았다! **{first_mismatch_idx}번**부터 순서가 꼬였습니다.")
    print("내 데이터셋 중간에 뭔가가 빠졌네요.")

# =========================================================
# [자동 해결] 팀장님 순서대로 yaml 내용 생성
# =========================================================
print("\n" + "="*60)
print("✅ 해결책: 아래 내용을 복사해서 'dataset.yaml' 파일에 덮어쓰세요!")
print("="*60)

new_names_list = []
for i in range(len(model_names)):
    new_names_list.append(model_names[i])

print(f"names: {new_names_list}")
print(f"nc: {len(model_names)}")

🕵️ 범인 색출 시작 (어디서부터 밀렸나?)...

ID    | 팀장 모델(Model)              | 내 데이터(YAML)               | 상태
20    | 콜리네이트연질캡슐 400mg           | 콜리네이트연질캡슐 400mg           | ✅
25    | 플라빅스정 75mg                | 플라빅스정 75mg                | ✅
30    | 자누비아정 50mg                | 자누비아정 50mg                | ✅
35    | 아모잘탄정 5/100mg             | 아모잘탄정 5/100mg             | ✅
40    | 트라젠타정(리나글립틴)              | 트라젠타정(리나글립틴)              | ✅
45    | 아질렉트정(라사길린메실산염)           | 아질렉트정(라사길린메실산염)           | ✅
50    | 글리틴정(콜린알포세레이트)            | 글리틴정(콜린알포세레이트)            | ✅
55    | 카발린캡슐 25mg                | 카발린캡슐 25mg                | ✅
------------------------------------------------------------
어라? 이름은 끝까지 똑같은데요? (그럼 마지막 57번째만 없는 겁니다)

✅ 해결책: 아래 내용을 복사해서 'dataset.yaml' 파일에 덮어쓰세요!
names: ['보령부스파정 5mg', '뮤테란캡슐 100mg', '일양하이트린정 2mg', '기넥신에프정(은행엽엑스)(수출용)', '무코스타정(레바미피드)(비매품)', '알드린정', '뉴로메드정(옥시라세탐)', '에어탈정(아세클로페낙)', '리렉스펜정 300mg/PTP', '아빌리파이정 10mg', '다보타민큐정 10mg/병', '써스펜8시간이알서방정 650mg', '에빅사정(메만틴염산염)(비매품)', 

In [19]:
import yaml
import os
import glob
from ultralytics import YOLO

# =========================================================
# [설정] 경로 확인
# =========================================================
model_path = '/content/drive/MyDrive/AI_Team_Project/best(0.985).pt'
root_dir = '/content/yolo_pill_ds' # 압축 푼 곳
# =========================================================

# 1. 모델에서 진짜 이름표(names) 가져오기
print("🧠 모델의 뇌(Metadata)에서 이름표를 추출합니다...")
model = YOLO(model_path)
true_names = model.names  # {0: 'Name1', 1: 'Name2', ...}
true_nc = len(true_names)

# 딕셔너리를 리스트로 변환 (ID 순서대로)
clean_names = [true_names[i] for i in range(true_nc)]

# 2. 고칠 yaml 파일 찾기
found_yamls = glob.glob(os.path.join(root_dir, "**", "data.yaml"), recursive=True) + \
              glob.glob(os.path.join(root_dir, "**", "dataset.yaml"), recursive=True)

if not found_yamls:
    print("❌ 수정할 yaml 파일을 찾지 못했습니다.")
else:
    target_yaml = found_yamls[0]
    print(f"📂 수정할 파일: {target_yaml}")

    # 3. 기존 파일 읽기 (경로 정보 등을 유지하기 위해)
    with open(target_yaml, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)

    # 4. 내용 업데이트
    config['names'] = clean_names
    config['nc'] = true_nc

    # 5. 덮어쓰기
    with open(target_yaml, 'w', encoding='utf-8') as f:
        # allow_unicode=True 를 해줘야 한글이 안 깨지고 잘 저장됩니다.
        yaml.dump(config, f, allow_unicode=True, sort_keys=False)

    print("-" * 50)
    print(f"✅ 수정 완료!")
    print(f" - 클래스 개수: {true_nc}개로 변경")
    print(f" - 이름표: 모델의 것과 100% 동기화됨")
    print("-" * 50)
    print("🚀 이제 아까 그 '오답 노트 코드'를 다시 돌려보세요!")

🧠 모델의 뇌(Metadata)에서 이름표를 추출합니다...
📂 수정할 파일: /content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/data.yaml
--------------------------------------------------
✅ 수정 완료!
 - 클래스 개수: 57개로 변경
 - 이름표: 모델의 것과 100% 동기화됨
--------------------------------------------------
🚀 이제 아까 그 '오답 노트 코드'를 다시 돌려보세요!


In [27]:
import os
import glob
import yaml
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

# =========================================================
# 설정 (이미 확인된 경로들)
# =========================================================
model_path = '/content/drive/MyDrive/AI_Team_Project/best(0.985).pt'
dataset_yaml_path = '/content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/data.yaml'
train_img_dir = '/content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/images/train'
train_label_dir = '/content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/labels/train'
# =========================================================

# 1. 이름표 동기화
model = YOLO(model_path)
model_names = model.names
with open(dataset_yaml_path, 'r') as f:
    dataset_names = yaml.safe_load(f)['names']

img_files = glob.glob(os.path.join(train_img_dir, "*.jpg")) + glob.glob(os.path.join(train_img_dir, "*.png"))
print(f"🚀 {len(img_files)}장의 훈련 데이터를 '이름' 기준으로 정밀 재검토합니다.")

real_hard_cases = []

for img_path in tqdm(img_files):
    file_name = os.path.basename(img_path)
    label_file = os.path.join(train_label_dir, os.path.splitext(file_name)[0] + ".txt")
    if not os.path.exists(label_file): continue

    # 정답 이름들 싹 다 가져오기 (한 이미지에 여러 알약 대비)
    gt_names = []
    with open(label_file, 'r') as f:
        for line in f:
            idx = int(line.split()[0])
            if idx < len(dataset_names):
                gt_names.append(dataset_names[idx])

    # 모델 예측
    results = model.predict(img_path, verbose=False, conf=0.1) # 어느 정도 확신 있는 것만

    for box in results[0].boxes:
        p_idx = int(box.cls.item())
        p_name = model_names[p_idx]
        p_conf = box.conf.item()

        # [핵심] 모델이 예측한 이름이 정답 이름 목록에 없으면 진짜 오답!
        if p_name not in gt_names:
            real_hard_cases.append({
                "파일명": file_name,
                "구분": "진짜오답",
                "예측이름": p_name,
                "확신도(%)": round(p_conf * 100, 2)
            })
        # 맞히긴 했는데 80% 미만인 경우 (mAP 깎아먹는 주범)
        elif p_conf < 0.8:
            real_hard_cases.append({
                "파일명": file_name,
                "구분": "자신없음",
                "예측이름": p_name,
                "확신도(%)": round(p_conf * 100, 2)
            })

if real_hard_cases:
    df = pd.DataFrame(real_hard_cases)
    df.to_csv("real_train_errors.csv", index=False, encoding='utf-8-sig')
    print(f"\n✅ 분석 완료! 진짜 문제는 {len(df)}개입니다. (real_train_errors.csv)")
    print(df['구분'].value_counts())
else:
    print("\n🎉 완벽합니다. 훈련 데이터에서 오답이 하나도 없습니다.")

🚀 6437장의 훈련 데이터를 '이름' 기준으로 정밀 재검토합니다.


100%|██████████| 6437/6437 [04:31<00:00, 23.74it/s]



✅ 분석 완료! 진짜 문제는 10439개입니다. (real_train_errors.csv)
구분
진짜오답    10398
자신없음       41
Name: count, dtype: int64


In [ ]:
import os
import glob
import yaml
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

# =========================================================
# 설정 (이미 확인된 경로들)
# =========================================================
model_path = '/content/drive/MyDrive/AI_Team_Project/runs/phase2_final/weights/best.pt'
dataset_yaml_path = '/content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/data.yaml'
train_img_dir = '/content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/images/train'
train_label_dir = '/content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/labels/train'
# =========================================================

# 1. 이름표 동기화
model = YOLO(model_path)
model_names = model.names
with open(dataset_yaml_path, 'r') as f:
    dataset_names = yaml.safe_load(f)['names']

img_files = glob.glob(os.path.join(train_img_dir, "*.jpg")) + glob.glob(os.path.join(train_img_dir, "*.png"))
print(f"🚀 {len(img_files)}장의 훈련 데이터를 '이름' 기준으로 정밀 재검토합니다.")

real_hard_cases = []

for img_path in tqdm(img_files):
    file_name = os.path.basename(img_path)
    label_file = os.path.join(train_label_dir, os.path.splitext(file_name)[0] + ".txt")
    if not os.path.exists(label_file): continue

    # 정답 이름들 싹 다 가져오기 (한 이미지에 여러 알약 대비)
    gt_names = []
    with open(label_file, 'r') as f:
        for line in f:
            idx = int(line.split()[0])
            if idx < len(dataset_names):
                gt_names.append(dataset_names[idx])

    # 모델 예측
    results = model.predict(img_path, verbose=False, conf=0.1) # 어느 정도 확신 있는 것만

    for box in results[0].boxes:
        p_idx = int(box.cls.item())
        p_name = model_names[p_idx]
        p_conf = box.conf.item()

        # [핵심] 모델이 예측한 이름이 정답 이름 목록에 없으면 진짜 오답!
        if p_name not in gt_names:
            real_hard_cases.append({
                "파일명": file_name,
                "구분": "진짜오답",
                "예측이름": p_name,
                "확신도(%)": round(p_conf * 100, 2)
            })
        # 맞히긴 했는데 80% 미만인 경우 (mAP 깎아먹는 주범)
        elif p_conf < 0.8:
            real_hard_cases.append({
                "파일명": file_name,
                "구분": "자신없음",
                "예측이름": p_name,
                "확신도(%)": round(p_conf * 100, 2)
            })

if real_hard_cases:
    df = pd.DataFrame(real_hard_cases)
    df.to_csv("real_train_errors(2).csv", index=False, encoding='utf-8-sig')
    print(f"\n✅ 분석 완료! 진짜 문제는 {len(df)}개입니다. (real_train_errors.csv)")
    print(df['구분'].value_counts())
else:
    print("\n🎉 완벽합니다. 훈련 데이터에서 오답이 하나도 없습니다.")

In [28]:
import pandas as pd

# 아까 생성된 csv 파일 읽기
df = pd.read_csv('real_train_errors.csv')

# 1. 진짜 틀린 놈 (정답 != 예측)
wrong_count = len(df[df['상태'] == '오답'])

# 2. 맞혔는데 확신도만 낮은 놈 (상태 == 불안한정답)
low_conf_count = len(df[df['상태'] == '불안한정답'])

print("="*50)
print(f"📊 5,032개 '범인'의 정체 폭로")
print("-" * 50)
print(f"❌ 진짜 틀린 놈 (Wrong): {wrong_count}개")
print(f"⚠️ 맞혔는데 자신 없는 놈 (Low-Conf): {low_conf_count}개")
print("="*50)

if wrong_count < 100:
    print("😎 거봐요. 진짜 틀린 건 얼마 안 되죠? 모델은 멀쩡합니다.")
else:
    print("🤔 오답이 좀 있네요? 이 놈들만 모아서 분석해볼 가치가 있습니다.")

KeyError: '상태'

In [30]:
import os

# 훈련 데이터 라벨 하나만 샘플로 찍어보기
sample_label = '/content/yolo_pill_ds/YOLO_Dataset_New_Smart_Strict_7787(2)/labels/train'
files = os.listdir(sample_label)

if files:
    with open(os.path.join(sample_label, files[0]), 'r') as f:
        print("파일에 적힌 번호:", f.readline().split()[0])

    # 모델이 아는 0번 이름
    print("모델이 아는 0번:", model.names[0])

파일에 적힌 번호: 55
모델이 아는 0번: 보령부스파정 5mg
